# DuckDB + SLayer, in Python

DuckDB reads a **48 KB CSV straight off a CDN** over `httpfs`; SLayer auto-ingests its schema and answers queries about it. Nothing is copied locally — the DuckDB view points at the URL, and every query reaches back over the wire.

**Prerequisites:** `pip install motley-slayer` (DuckDB ships with it).

## 1. Point DuckDB at the file online

We create a **view** over the remote CSV inside a file-backed DuckDB database. DuckDB auto-installs the `httpfs` extension and streams the file; no rows land on disk. We close the connection right after — DuckDB won't share a read-write file across connections, and SLayer opens its own.

In [1]:
import shutil
import warnings
from pathlib import Path

import duckdb
import pandas as pd

# duckdb-engine warns that it can't reflect indices — irrelevant here.
warnings.filterwarnings("ignore", message=".*reflection on indices.*")

from slayer.async_utils import run_sync
from slayer.client.slayer_client import SlayerClient
from slayer.core.models import DatasourceConfig
from slayer.engine.ingestion import ingest_datasource_idempotent
from slayer.storage.yaml_storage import YAMLStorage

CACHE = Path(".cache/python")
shutil.rmtree(CACHE, ignore_errors=True)
CACHE.mkdir(parents=True)

DB_PATH = (CACHE / "weather.duckdb").resolve()
CSV_URL = "https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv"

con = duckdb.connect(str(DB_PATH))
con.execute(f"CREATE OR REPLACE VIEW weather AS SELECT * FROM '{CSV_URL}'")
n_rows = con.sql("SELECT count(*) FROM weather").fetchone()[0]
assert n_rows == 1461, f"expected 1461 rows, got {n_rows}"
schema = con.sql("DESCRIBE weather").fetchdf()
con.close()
schema

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,precipitation,DOUBLE,YES,None,None,None
2,temp_max,DOUBLE,YES,None,None,None
3,temp_min,DOUBLE,YES,None,None,None
4,wind,DOUBLE,YES,None,None,None
5,weather,VARCHAR,YES,None,None,None


## 2. Auto-ingest the schema, SLayer-side

SLayer introspects the view — off the live URL — and builds a semantic model named `weather`, one column per field with a type. No hand-written model.

In [2]:
storage = YAMLStorage(base_dir=str(CACHE / "models"))
ds = DatasourceConfig(name="weather_db", type="duckdb", database=str(DB_PATH))
run_sync(storage.save_datasource(ds))

run_sync(ingest_datasource_idempotent(datasource=ds, storage=storage))

models = run_sync(storage.list_models(data_source="weather_db"))
assert "weather" in models, f"weather model not ingested; got {models}"

weather = run_sync(storage.get_model(name="weather", data_source="weather_db"))
pd.DataFrame([{"column": c.name, "type": c.type} for c in weather.columns])

,column,type
0,date,DATE
1,precipitation,DOUBLE
2,temp_max,DOUBLE
3,temp_min,DOUBLE
4,wind,DOUBLE
5,weather,TEXT


## 3. A warm-up query

Average high and day count per weather type — a plain grouped aggregation, to confirm the model answers questions. Aggregations are chosen at query time with colon syntax (`temp_max:avg`, `*:count`).

In [3]:
client = SlayerClient(storage=storage)

warmup = client.query_sync(
    {
        "source_model": "weather",
        "dimensions": ["weather"],
        "measures": [
            {"formula": "temp_max:avg", "name": "avg_high"},
            {"formula": "*:count", "name": "days"},
        ],
        "order": [{"column": "days", "direction": "desc"}],
    }
)
pd.DataFrame(warmup.data)

,weather.weather,weather.avg_high,weather.days
0,rain,13.454602,641
1,sun,19.861875,640
2,fog,16.757426,101
3,drizzle,15.926415,53
4,snow,5.573077,26


## 4. The hero query: a computed band and year-over-year, in one query

Two query-time computations at the month grain, in **one** query:

- **A band from the monthly total.** `month_type` is a labeled measure — `CASE WHEN precipitation:sum > 100 THEN 'rainy' ELSE 'dry' END` — tagging each month *rainy* or *dry* by its own rainfall total. `precipitation:sum` is evaluated per month, so the band comes straight off the aggregate.
- **Change versus the same month last year.** `time_shift(precipitation:sum, -1, 'year')` is the rainfall total twelve months back; subtracting gives the year-over-year change. It's calendar-aware, so December compares to the previous December.

The first year (2012) has nothing to look back to, so its year-over-year values are null — `time_shift` reaches outside the result when the data exists, and is null only when it genuinely doesn't.

In [4]:
hero = {
    "source_model": "weather",
    "time_dimensions": [{"dimension": "date", "granularity": "month"}],
    "measures": [
        {
            "formula": "CASE WHEN precipitation:sum > 100 THEN 'rainy' ELSE 'dry' END",
            "name": "month_type",
        },
        {"formula": "precipitation:sum", "name": "total_rain"},
        {
            "formula": "precipitation:sum - time_shift(precipitation:sum, -1, 'year')",
            "name": "total_rain_yoy",
        },
    ],
    "order": [{"column": "date", "direction": "asc"}],
}

result = client.query_sync(hero)
df = pd.DataFrame(result.data)

n_null_yoy = int(df["weather.total_rain_yoy"].isna().sum())
assert len(df) == 48, f"expected 48 monthly rows, got {len(df)}"
assert n_null_yoy == 12, f"expected 12 null YoY rows (first year), got {n_null_yoy}"
df

,weather.date,weather.month_type,weather.total_rain,weather.total_rain_yoy
0,2012-01-01,rainy,173.3,NaN
1,2012-02-01,dry,92.3,NaN
2,2012-03-01,rainy,183.0,NaN
3,2012-04-01,dry,68.1,NaN
4,2012-05-01,dry,52.2,NaN
5,2012-06-01,dry,75.1,NaN
6,2012-07-01,dry,26.3,NaN
7,2012-08-01,dry,0.0,NaN
8,2012-09-01,dry,0.9,NaN
9,2012-10-01,rainy,170.3,NaN


## 5. The SQL SLayer ran

Why use SLayer at all, instead of writing SQL directly? Well, here's the SQL corresponding to the above query. Which one do you think is easier for agents to write, or for humans to audit?

In [5]:
assert result.sql, "expected generated SQL on the response"
print(result.sql)

SELECT
    "weather.date",
    "weather.month_type",
    "weather.total_rain",
    "weather.total_rain_yoy"
FROM (
WITH base AS (
  SELECT
    DATE_TRUNC('MONTH', weather.date) AS "weather.date",
    CASE WHEN SUM(weather.precipitation) > 100 THEN 'rainy' ELSE 'dry' END AS "weather.month_type",
    CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.total_rain"
  FROM weather AS weather
  GROUP BY
    DATE_TRUNC('MONTH', weather.date)
), shifted__time_shift_inner AS (
  SELECT
    DATE_TRUNC('MONTH', weather.date) + INTERVAL 1 YEAR AS "weather.date",
    CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.total_rain"
  FROM weather AS weather
  GROUP BY
    DATE_TRUNC('MONTH', weather.date) + INTERVAL 1 YEAR
), sjoin__time_shift_inner AS (
  SELECT
    base."weather.date",
    base."weather.month_type",
    base."weather.total_rain",
    shifted__time_shift_inner."weather.total_rain" AS "weather._time_shift_inner"
  FROM base
  LEFT JOIN shifted__time_shift_inner
    ON base."wea

---

That's a semantic layer over a file on the internet, in a page of code. See the [command-line version](duckdb_cli_nb.ipynb) for the same demo driven entirely through the `slayer` CLI, the [multi-stage queries](../06_multistage_queries/multistage_queries.md) guide for the queries-as-models idea, and [formulas](../../concepts/formulas.md) for the full transform vocabulary.